# Tension Sweep: Full 0-100% Strategy Mapping

**Goal**: For every agreement range (0-10%, 10-20%, ... 90-100%),
find which strategy wins. This data designs Big Auto.

**Reuses cached embeddings from Batch A** — no re-encoding needed.

**Output**: One JSON with per-bin optimal strategy + ArguAna analysis

In [6]:
!pip install -q beir sentence-transformers rank-bm25 numpy pytrec-eval-terrier
!nvidia-smi | head -5 || echo 'No GPU'

Sat Mar 28 03:55:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [7]:
import json
import os
import re
import time
from collections import defaultdict

import numpy as np
import pytrec_eval
import torch
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [8]:
# ── Fusion functions ──

def _norm(results):
    if not results:
        return {}
    vals = [s for _, s in results]
    mn, mx = min(vals), max(vals)
    rng = mx - mn if mx > mn else 1.0
    return {did: (s - mn) / rng for did, s in results}


def simple_rrf(b, d, k=5, bw=1.0, dw=1.2):
    scores = defaultdict(float)
    for rank, (did, _) in enumerate(b):
        scores[did] += bw / (k + rank + 1)
    for rank, (did, _) in enumerate(d):
        scores[did] += dw / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def riverbed_only(b, d, bw=0.8, dw=1.4):
    b_n, d_n = _norm(b), _norm(d)
    all_docs = set(b_n) | set(d_n)
    tw = bw + dw
    final = {
        did: (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / tw
        for did in all_docs
    }
    return sorted(final.items(), key=lambda x: x[1], reverse=True)


def riverbed_dense_only(b, d, dw=1.0):
    """Riverbed but ONLY preserve Dense scores, ignore BM25 scores.
    For ArguAna-type: BM25 scores are poison (same-side confidence).
    Still use BM25 ranks via RRF component, just don't trust its scores."""
    d_n = _norm(d)
    # RRF for rank component (BM25 contributes rank only)
    rrf_scores = defaultdict(float)
    for rank, (did, _) in enumerate(b):
        rrf_scores[did] += 1.0 / (5 + rank + 1)
    for rank, (did, _) in enumerate(d):
        rrf_scores[did] += 1.2 / (5 + rank + 1)
    # Normalize RRF
    rv = list(rrf_scores.values())
    r_mn, r_mx = min(rv), max(rv)
    r_rng = r_mx - r_mn if r_mx > r_mn else 1.0
    all_docs = set(rrf_scores) | set(d_n)
    final = {}
    for did in all_docs:
        r = (rrf_scores.get(did, 0) - r_mn) / r_rng
        s = d_n.get(did, 0)
        final[did] = 0.5 * r + 0.5 * s
    return sorted(final.items(), key=lambda x: x[1], reverse=True)


def rt_full(
    b, d, k_low=3, k_high=10, top_n=20,
    boost_max=1.2, score_w=0.5, bw=0.8, dw=1.4,
):
    b_set = {did for did, _ in b[:top_n]}
    d_set = {did for did, _ in d[:top_n]}
    union = b_set | d_set
    agreement = len(b_set & d_set) / len(union) if union else 0.0
    tension = 1.0 - agreement
    adaptive_k = max(1, int(k_low + (k_high - k_low) * tension))
    boost = 1.0 + (boost_max - 1.0) * agreement

    rrf_scores = defaultdict(float)
    presence = defaultdict(int)
    for rank, (did, _) in enumerate(b):
        rrf_scores[did] += bw / (adaptive_k + rank + 1)
        presence[did] += 1
    for rank, (did, _) in enumerate(d):
        rrf_scores[did] += dw / (adaptive_k + rank + 1)
        presence[did] += 1
    for did in rrf_scores:
        if presence[did] >= 2:
            rrf_scores[did] *= boost

    b_n, d_n = _norm(b), _norm(d)
    rv = list(rrf_scores.values())
    r_mn, r_mx = min(rv), max(rv)
    r_rng = r_mx - r_mn if r_mx > r_mn else 1.0
    tw = bw + dw

    all_docs = set(rrf_scores) | set(b_n) | set(d_n)
    final = {}
    for did in all_docs:
        r = (rrf_scores.get(did, 0) - r_mn) / r_rng
        s = (bw * b_n.get(did, 0) + dw * d_n.get(did, 0)) / tw
        final[did] = (1 - score_w) * r + score_w * s
    return sorted(final.items(), key=lambda x: x[1], reverse=True)


def measure_agreement(b, d, top_n=20):
    b_set = {did for did, _ in b[:top_n]}
    d_set = {did for did, _ in d[:top_n]}
    union = b_set | d_set
    return len(b_set & d_set) / len(union) if union else 0.0


STRATEGIES = {
    "dense_only": lambda b, d: d,
    "rrf": lambda b, d: simple_rrf(b, d),
    "riverbed": lambda b, d: riverbed_only(b, d),
    "riverbed_dense": lambda b, d: riverbed_dense_only(b, d),
    "rt_full": lambda b, d: rt_full(b, d),
}

print(f"{len(STRATEGIES)} strategies ready")

5 strategies ready


In [9]:
# ── Load datasets + BM25 + embeddings (reuses Batch A caches) ──

BEIR_BASE = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets"
DATASETS = {
    "scifact": f"{BEIR_BASE}/scifact.zip",
    "nfcorpus": f"{BEIR_BASE}/nfcorpus.zip",
    "arguana": f"{BEIR_BASE}/arguana.zip",
    "scidocs": f"{BEIR_BASE}/scidocs.zip",
    "fiqa": f"{BEIR_BASE}/fiqa.zip",
}

BASE_DIR = "datasets"
os.makedirs(BASE_DIR, exist_ok=True)
MODEL_ID = "intfloat/e5-base-unsupervised"

def tokenize(text):
    return re.findall(r'\w+', text.lower())

print("Loading model...")
model = SentenceTransformer(MODEL_ID, device=DEVICE)

all_data = {}
for ds_name, url in DATASETS.items():
    print(f"\nLoading {ds_name}...")
    data_path = os.path.join(BASE_DIR, ds_name)
    if not os.path.isdir(data_path):
        data_path = util.download_and_unzip(url, BASE_DIR)
    corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

    # BM25
    doc_ids = list(corpus.keys())
    tokenized = [
        tokenize(f"{corpus[did].get('title','')} {corpus[did].get('text','')}") 
        for did in doc_ids
    ]
    bm25 = BM25Okapi(tokenized)

    # Dense (from cache)
    tag = MODEL_ID.replace('/', '_').replace('-', '_')
    cache_path = os.path.join(BASE_DIR, f".cache_{ds_name}_{tag}.npz")
    if os.path.exists(cache_path):
        embs = np.load(cache_path)["embs"]
        print(f"  Cache hit: {embs.shape}")
    else:
        print(f"  Encoding {len(corpus)} docs...")
        texts = [
            f"passage: {corpus[did].get('title','')} {corpus[did].get('text','')}".strip()
            for did in doc_ids
        ]
        embs = model.encode(texts, normalize_embeddings=True, batch_size=128, show_progress_bar=True)
        np.savez_compressed(cache_path, embs=embs)

    all_data[ds_name] = {
        "corpus": corpus, "queries": queries, "qrels": qrels,
        "bm25": bm25, "doc_ids": doc_ids, "embs": embs,
    }
    print(f"  {ds_name}: {len(corpus)} docs, {len(queries)} queries")

print("\nAll loaded")

Loading model...

Loading scifact...


  0%|          | 0/5183 [00:00<?, ?it/s]

  Cache hit: (5183, 768)
  scifact: 5183 docs, 300 queries

Loading nfcorpus...


  0%|          | 0/3633 [00:00<?, ?it/s]

  Cache hit: (3633, 768)
  nfcorpus: 3633 docs, 323 queries

Loading arguana...


...

  Cache hit: (8674, 768)
  arguana: 8674 docs, 1406 queries

Loading scidocs...


  0%|          | 0/25657 [00:00<?, ?it/s]

  Cache hit: (25657, 768)
  scidocs: 25657 docs, 1000 queries

Loading fiqa...


  0%|          | 0/57638 [00:00<?, ?it/s]

  Cache hit: (57638, 768)
  fiqa: 57638 docs, 648 queries

All loaded


In [10]:
# ── Per-query evaluation: agreement + nDCG per strategy ──

def ndcg_single(ranked_ids, relevant, k=10):
    """Single-query nDCG (pytrec_eval is batch-only, need this for per-query)."""
    import math
    dcg = sum(
        relevant.get(did, 0) / math.log2(i + 2)
        for i, did in enumerate(ranked_ids[:k])
    )
    ideal = sorted(relevant.values(), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


per_query_data = {}  # {ds: [{agreement, strat_scores}, ...]}

for ds_name, d in all_data.items():
    print(f"\n{ds_name}: evaluating per-query...")
    queries = d["queries"]
    qrels = d["qrels"]
    bm25 = d["bm25"]
    doc_ids = d["doc_ids"]
    embs = d["embs"]

    rows = []
    for qi, (qid, qt) in enumerate(queries.items()):
        rel = {did: r for did, r in qrels.get(qid, {}).items() if r > 0}
        if not rel:
            continue

        # BM25
        bm25_scores = bm25.get_scores(tokenize(qt))
        top_idx = bm25_scores.argsort()[-100:][::-1]
        bm25_res = [
            (doc_ids[i], float(bm25_scores[i]))
            for i in top_idx if bm25_scores[i] > 0
        ]

        # Dense
        q_emb = model.encode(["query: " + qt], normalize_embeddings=True)
        sims = (embs @ q_emb.T).flatten()
        idx = np.argsort(sims)[::-1][:100]
        dense_res = [(doc_ids[i], float(sims[i])) for i in idx]

        agr = measure_agreement(bm25_res, dense_res)

        # Evaluate each strategy
        strat_scores = {}
        for sname, sfn in STRATEGIES.items():
            fused = sfn(bm25_res, dense_res)
            ranked = [did for did, _ in fused[:100]]
            strat_scores[sname] = ndcg_single(ranked, rel, k=10)

        rows.append({"qid": qid, "agreement": agr, "scores": strat_scores})

        if (qi + 1) % 300 == 0:
            print(f"  {qi + 1}/{len(queries)}")

    per_query_data[ds_name] = rows
    print(f"  {len(rows)} queries evaluated")

print("\nAll per-query evaluation done")


scifact: evaluating per-query...
  300/300
  300 queries evaluated

nfcorpus: evaluating per-query...
  300/323
  323 queries evaluated

arguana: evaluating per-query...
  300/1406
  600/1406
  900/1406
  1200/1406
  1406 queries evaluated

scidocs: evaluating per-query...
  300/1000
  600/1000
  900/1000
  1000 queries evaluated

fiqa: evaluating per-query...
  300/648
  600/648
  648 queries evaluated

All per-query evaluation done


In [11]:
# ── Bin by agreement and find optimal strategy per bin ──

BINS = [(i/10, (i+1)/10) for i in range(10)]  # 0-10%, 10-20%, ..., 90-100%
BIN_LABELS = [f"{int(lo*100)}-{int(hi*100)}%" for lo, hi in BINS]

strat_names = list(STRATEGIES.keys())

# Per-dataset, per-bin analysis
bin_analysis = {}

for ds_name, rows in per_query_data.items():
    ds_bins = []
    for bi, (lo, hi) in enumerate(BINS):
        # Queries in this agreement bin
        in_bin = [
            r for r in rows
            if lo <= r["agreement"] < hi or (hi == 1.0 and r["agreement"] == 1.0)
        ]
        if not in_bin:
            ds_bins.append({
                "bin": BIN_LABELS[bi], "count": 0,
                "best": "N/A", "scores": {},
            })
            continue

        # Average nDCG per strategy in this bin
        avg_scores = {}
        for sn in strat_names:
            vals = [r["scores"][sn] for r in in_bin]
            avg_scores[sn] = round(sum(vals) / len(vals), 4)

        best = max(avg_scores, key=avg_scores.get)
        ds_bins.append({
            "bin": BIN_LABELS[bi],
            "count": len(in_bin),
            "best": best,
            "scores": avg_scores,
        })
    bin_analysis[ds_name] = ds_bins

# Print results
for ds_name in DATASETS:
    print(f"\n{'=' * 80}")
    print(f"{ds_name.upper()} — Strategy by Agreement Bin")
    print(f"{'=' * 80}")
    header = f"{'Bin':<10} {'N':>5}"
    for sn in strat_names:
        header += f" {sn:>14}"
    header += f" {'BEST':>14}"
    print(header)
    print("-" * len(header))

    for b in bin_analysis[ds_name]:
        if b["count"] == 0:
            row = f"{b['bin']:<10} {0:>5}   (no queries)"
        else:
            row = f"{b['bin']:<10} {b['count']:>5}"
            for sn in strat_names:
                s = b["scores"].get(sn, 0)
                marker = " *" if sn == b["best"] else "  "
                row += f" {s:>12.4f}{marker}"
            row += f" {b['best']:>14}"
        print(row)


SCIFACT — Strategy by Agreement Bin
Bin            N     dense_only            rrf       riverbed riverbed_dense        rt_full           BEST
----------------------------------------------------------------------------------------------------------
0-10%         59       0.5768         0.6303 *       0.6234         0.6196         0.6257              rrf
10-20%       112       0.7821         0.7835 *       0.7831         0.7828         0.7808              rrf
20-30%        73       0.8082         0.8117         0.8260 *       0.8125         0.8230         riverbed
30-40%        37       0.6994         0.6885         0.7240 *       0.7116         0.7150         riverbed
40-50%        11       0.8238         0.8811         0.9028         0.8874         0.9091 *        rt_full
50-60%         3       0.6081         0.6733 *       0.6696         0.6081         0.6696              rrf
60-70%         3       0.6570 *       0.5065         0.6281         0.6450         0.6311       dense_only


In [12]:
# ── Cross-dataset consensus: what strategy for each bin? ──

print("\n" + "=" * 60)
print("CROSS-DATASET CONSENSUS: Best Strategy per Agreement Bin")
print("=" * 60)

# For each bin, vote across datasets
consensus = []
for bi, label in enumerate(BIN_LABELS):
    votes = defaultdict(int)
    weighted_scores = defaultdict(float)
    total_queries = 0

    for ds_name in DATASETS:
        b = bin_analysis[ds_name][bi]
        if b["count"] == 0:
            continue
        votes[b["best"]] += 1
        # Weight by number of queries
        for sn, sc in b["scores"].items():
            weighted_scores[sn] += sc * b["count"]
        total_queries += b["count"]

    if total_queries == 0:
        consensus.append({"bin": label, "best": "N/A", "count": 0})
        continue

    # Weighted best
    for sn in weighted_scores:
        weighted_scores[sn] /= total_queries
    weighted_best = max(weighted_scores, key=weighted_scores.get)

    consensus.append({
        "bin": label,
        "count": total_queries,
        "weighted_best": weighted_best,
        "vote_best": max(votes, key=votes.get),
        "votes": dict(votes),
        "weighted_scores": {k: round(v, 4) for k, v in weighted_scores.items()},
    })

    print(
        f"{label:<10} N={total_queries:>5}  "
        f"weighted_best={weighted_best:<14}  "
        f"votes={dict(votes)}"
    )

# Derive Big Auto routing table
print("\n" + "=" * 60)
print("BIG AUTO ROUTING TABLE")
print("=" * 60)
for c in consensus:
    if c["count"] > 0:
        best = c["weighted_best"]
        print(f"  agreement {c['bin']:<10} → {best}")


CROSS-DATASET CONSENSUS: Best Strategy per Agreement Bin
0-10%      N=  949  weighted_best=riverbed_dense  votes={'rrf': 2, 'rt_full': 1, 'riverbed_dense': 2}
10-20%     N=  915  weighted_best=riverbed_dense  votes={'rrf': 1, 'rt_full': 1, 'riverbed_dense': 3}
20-30%     N=  804  weighted_best=riverbed_dense  votes={'riverbed': 3, 'riverbed_dense': 2}
30-40%     N=  407  weighted_best=riverbed        votes={'riverbed': 2, 'rt_full': 1, 'rrf': 2}
40-50%     N=  302  weighted_best=rrf             votes={'rt_full': 2, 'riverbed': 1, 'rrf': 1, 'dense_only': 1}
50-60%     N=  117  weighted_best=rrf             votes={'rrf': 5}
60-70%     N=  131  weighted_best=rrf             votes={'dense_only': 1, 'riverbed': 2, 'rrf': 1}
70-80%     N=   35  weighted_best=rrf             votes={'dense_only': 2, 'rrf': 1}
80-90%     N=    9  weighted_best=riverbed        votes={'riverbed': 1}
90-100%    N=    8  weighted_best=dense_only      votes={'riverbed_dense': 1, 'dense_only': 1}

BIG AUTO ROUTING T

In [13]:
# ── Simulate Big Auto: route each query to bin-optimal strategy ──

print("\n" + "=" * 60)
print("BIG AUTO SIMULATION")
print("=" * 60)

# Build routing table from consensus
routing = {}
for c in consensus:
    if c["count"] > 0:
        routing[c["bin"]] = c["weighted_best"]


def get_bin_label(agreement):
    bi = min(int(agreement * 10), 9)
    return BIN_LABELS[bi]


for ds_name, rows in per_query_data.items():
    # Auto: pick per-query best strategy based on agreement bin
    auto_ndcgs = []
    static_ndcgs = {sn: [] for sn in strat_names}

    for r in rows:
        bl = get_bin_label(r["agreement"])
        best_strat = routing.get(bl, "riverbed")
        auto_ndcgs.append(r["scores"][best_strat])

        for sn in strat_names:
            static_ndcgs[sn].append(r["scores"][sn])

    auto_avg = sum(auto_ndcgs) / len(auto_ndcgs)
    print(f"\n{ds_name}:")
    print(f"  Big Auto:     {auto_avg:.4f}")
    for sn in strat_names:
        avg = sum(static_ndcgs[sn]) / len(static_ndcgs[sn])
        delta = auto_avg - avg
        marker = " ← current best" if sn == max(
            static_ndcgs, key=lambda s: sum(static_ndcgs[s])
        ) else ""
        print(f"  {sn:<15} {avg:.4f} (Auto Δ={delta:+.4f}){marker}")


BIG AUTO SIMULATION

scifact:
  Big Auto:     0.7511
  dense_only      0.7371 (Auto Δ=+0.0140)
  rrf             0.7494 (Auto Δ=+0.0017)
  riverbed        0.7576 (Auto Δ=-0.0065) ← current best
  riverbed_dense  0.7510 (Auto Δ=+0.0000)
  rt_full         0.7557 (Auto Δ=-0.0046)

nfcorpus:
  Big Auto:     0.3635
  dense_only      0.3582 (Auto Δ=+0.0053)
  rrf             0.3604 (Auto Δ=+0.0031)
  riverbed        0.3639 (Auto Δ=-0.0004)
  riverbed_dense  0.3637 (Auto Δ=-0.0002)
  rt_full         0.3666 (Auto Δ=-0.0031) ← current best

arguana:
  Big Auto:     0.3364
  dense_only      0.3174 (Auto Δ=+0.0191)
  rrf             0.3348 (Auto Δ=+0.0016) ← current best
  riverbed        0.3285 (Auto Δ=+0.0080)
  riverbed_dense  0.3331 (Auto Δ=+0.0033)
  rt_full         0.3318 (Auto Δ=+0.0046)

scidocs:
  Big Auto:     0.2143
  dense_only      0.2110 (Auto Δ=+0.0033)
  rrf             0.2055 (Auto Δ=+0.0089)
  riverbed        0.2116 (Auto Δ=+0.0027)
  riverbed_dense  0.2139 (Auto Δ=+0.0005) ← c

In [14]:
# ── Save ONE JSON ──

output = {
    "experiment": "tension_sweep_full_0_100",
    "model": MODEL_ID,
    "evaluator": "pytrec_eval-compatible (per-query nDCG)",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "bin_analysis": bin_analysis,
    "consensus": consensus,
    "routing_table": routing,
}

print(json.dumps(output, indent=2))

{
  "experiment": "tension_sweep_full_0_100",
  "model": "intfloat/e5-base-unsupervised",
  "evaluator": "pytrec_eval-compatible (per-query nDCG)",
  "timestamp": "2026-03-28 04:19:11",
  "bin_analysis": {
    "scifact": [
      {
        "bin": "0-10%",
        "count": 59,
        "best": "rrf",
        "scores": {
          "dense_only": 0.5768,
          "rrf": 0.6303,
          "riverbed": 0.6234,
          "riverbed_dense": 0.6196,
          "rt_full": 0.6257
        }
      },
      {
        "bin": "10-20%",
        "count": 112,
        "best": "rrf",
        "scores": {
          "dense_only": 0.7821,
          "rrf": 0.7835,
          "riverbed": 0.7831,
          "riverbed_dense": 0.7828,
          "rt_full": 0.7808
        }
      },
      {
        "bin": "20-30%",
        "count": 73,
        "best": "riverbed",
        "scores": {
          "dense_only": 0.8082,
          "rrf": 0.8117,
          "riverbed": 0.826,
          "riverbed_dense": 0.8125,
          "rt_full"